<a href="https://colab.research.google.com/github/sreent/machine-learning/blob/main/Lectures/CNN%20Lab%20with%20Keras.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# CNN Lab with Keras

## **Table of Contents**

1. [Preliminaries & Setup](#section0)  
2. [Introduction & Key Concepts](#section1)  
   - 2.1 [Neural Network Fundamentals](#section1_1)  
   - 2.2 [Manual Convolution Filter Demonstration](#section1_2)  
   - 2.3 [Why CNNs vs. Dense Networks?](#section1_3)  
   - 2.4 [Mini-Quiz #1](#section1_4)  
3. [Part I: MNIST — Dense vs. CNN](#section2)  
   - 3.1 [Overview & Loading MNIST](#section2_1)  
   - 3.2 [Build & Train a Dense Network (Baseline)](#section2_2)  
   - 3.3 [Build & Train a CNN on MNIST](#section2_3)  
   - 3.4 [Visualizing CNN Filters & Activations](#section2_4)  
   - 3.5 [Try-It-Yourself (TIY)](#section2_5)  
   - 3.6 [Mini-Quiz #2](#section2_6)  
4. [Part II: CIFAR-10 — Data Augmentation](#section3)  
   - 4.1 [Loading & Exploring CIFAR-10](#section3_1)  
   - 4.1.1 [Data Augmentation Demonstration](#section3_1_1)  
   - 4.2 [CNN with Data Augmentation](#section3_2)  
   - 4.3 [Try-It-Yourself (TIY)](#section3_3)  
   - 4.4 [Mini-Quiz #3](#section3_4)  
5. [Part III: Transfer Learning (Cats vs. Dogs)](#section4)  
   - 5.1 [Why Transfer Learning?](#section4_1)  
   - 5.2 [Loading Cats vs. Dogs from TFDS](#section4_2)  
   - 5.3 [Feature Extraction & Fine-Tuning](#section4_3)  
   - 5.4 [Try-It-Yourself (TIY)](#section4_4)  
   - 5.5 [Mini-Quiz #4](#section4_5)  
6. [Optional: TensorBoard Demonstration](#section5)  
7. [Optional: Grad-CAM Demonstration](#section6)  
8. [Hyperparameter Tuning Tips](#section7)  
9. [Reflection & Summary](#section8)  
10. [Additional Best Practices](#section9)  
11. [Common Pitfalls & Solutions](#section10)  
12. [Wrap-Up & Further Resources](#section11)


<a name="section0"></a>
## **1. Preliminaries & Setup**

**For Google Colab**:
1. **Open** [Colab](https://colab.research.google.com).  
2. **Set GPU**: Runtime → Change runtime type → Hardware accelerator = GPU.  
3. **Install** packages:

In [ ]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt

!pip install tensorflow-datasets --quiet
print("TensorFlow version:", tf.__version__)

device_list = tf.config.list_physical_devices('GPU')
print("GPUs Available:", device_list)

<a name="section1"></a>
## **2. Introduction & Key Concepts**

<a name="section1_1"></a>
### 2.1 Neural Network Fundamentals
- **Neuron**: Computes $ \sigma(\mathbf{w} \cdot \mathbf{x} + b) $.  
- **Layers**: Dense, Convolutional, Pooling, etc.  
- **Forward & Backprop**: Predictions compare to ground truth to compute loss → update weights.

<a name="section1_2"></a>
### 2.2 Manual Convolution Filter Demonstration

**Show** how edge detection or blur works via a **manual** convolution of an image.  

In [ ]:
import requests, os
import cv2
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import convolve2d

# Download a sample grayscale image (Lena)
url = "https://boofcv.org/images/f/fe/Original_lena512.jpg"
img_path = "lena.jpg"
if not os.path.exists(img_path):
    r = requests.get(url)
    with open(img_path, "wb") as f:
        f.write(r.content)

img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
print("Original shape:", img.shape)

# Define filters
vertical_filter = np.array([
    [-1,  0,  1],
    [-1,  0,  1],
    [-1,  0,  1]
], dtype=np.float32)

horizontal_filter = np.array([
    [ 1,  1,  1],
    [ 0,  0,  0],
    [-1, -1, -1]
], dtype=np.float32)

corner_filter = np.array([
    [-1, -1, -1],
    [-1,  8, -1],
    [-1, -1, -1]
], dtype=np.float32)

# NEW: Sharpen filter
sharpen_filter = np.array([
    [ 0, -1,  0],
    [-1,  5, -1],
    [ 0, -1,  0]
], dtype=np.float32)

# NEW: Gaussian blur filter
gaussian_filter = np.array([
    [1, 2, 1],
    [2, 4, 2],
    [1, 2, 1]
], dtype=np.float32)/16.0

# Convolve
vertical_edges   = convolve2d(img, vertical_filter,  mode='same', boundary='symm')
horizontal_edges = convolve2d(img, horizontal_filter,mode='same', boundary='symm')
corner_response  = convolve2d(img, corner_filter,    mode='same', boundary='symm')
sharpened_img    = convolve2d(img, sharpen_filter,   mode='same', boundary='symm')
blurred_img      = convolve2d(img, gaussian_filter,  mode='same', boundary='symm')

# Plot results
fig, axs = plt.subplots(2, 3, figsize=(16,10))
axs[0,0].imshow(img, cmap='gray')
axs[0,0].set_title("Original"); axs[0,0].axis('off')

axs[0,1].imshow(vertical_edges, cmap='gray')
axs[0,1].set_title("Vertical Edges"); axs[0,1].axis('off')

axs[0,2].imshow(horizontal_edges, cmap='gray')
axs[0,2].set_title("Horizontal Edges"); axs[0,2].axis('off')

axs[1,0].imshow(corner_response, cmap='gray')
axs[1,0].set_title("Corner Filter"); axs[1,0].axis('off')

axs[1,1].imshow(sharpened_img, cmap='gray')
axs[1,1].set_title("Sharpen Filter"); axs[1,1].axis('off')

axs[1,2].imshow(blurred_img, cmap='gray')
axs[1,2].set_title("Gaussian Blur"); axs[1,2].axis('off')

plt.show()


Each output image corresponds to your original grayscale Lena image passed through a different 3×3 filter—so they each highlight or alter features in different ways:

1. **Vertical Edges**  
   - This filter responds strongly where there’s a **vertical intensity gradient**—i.e., columns in the image that change from dark to light or vice versa.  
   - Hence you see the outline of Lena’s hat, face, and background edges emphasized primarily where vertical transitions occur.

2. **Horizontal Edges**  
   - Similarly, this filter picks up **horizontal gradients**, so horizontal edges of the hat brim, shoulders, or background lines become most visible.  
   - Vertical transitions are largely ignored, making vertical edges appear less bright.

3. **Corner Filter** (Laplacian‐like kernel)  
   - This kernel is sensitive to **rapid intensity changes** in all directions—often including corners or small point‐like features.  
   - You’ll see an outline of Lena’s entire face and hat, but also more faint lines or corners that might not show up as strongly in the simple vertical/horizontal filters.

4. **Sharpen Filter**  
   - Emphasizes edges and local contrast by adding a portion of the pixel’s difference from its neighbors back to the pixel itself.  
   - So the overall image remains recognizable but with more pronounced edges/contrast—almost like an “unsharp mask” effect from classic image editing.

5. **Gaussian Blur**  
   - Smooths out local variations by taking a weighted average of each pixel’s neighbors, effectively **reducing noise and fine detail**.  
   - The result is a softer, blurred version of the original image, with edges and details less distinct.

### **Interpretation**

- **Edge Detectors (vertical/horizontal)**: Show you where the image has strong intensity gradients in specific directions.  
- **Corner/Laplacian**: Highlights points or areas where intensity changes sharply in all directions (often called corners).  
- **Sharpen**: Increases local contrast, making details pop more.  
- **Gaussian Blur**: Smooths or softens the image by averaging pixels in a Gaussian‐weighted neighborhood.

Together, these filters illustrate several common operations in image processing, each revealing or modifying different aspects of the same Lena image.

<a name="section1_3"></a>
### 2.3 Why CNNs vs. Dense Networks?

- **Dense**: Flatten images, ignoring local spatial structure, leading to a large parameter count.  
- **CNN**: Applies shared filters across the image, capturing edges & shapes more effectively with fewer parameters.

<a name="section1_4"></a>
### 2.4 Mini-Quiz #1

1. *Which operation do we use to detect edges in an image?*  
   A) **Convolution**  
   B) Fully connected layers  
   C) Batch normalization  

2. *True or False?* Convolution filters in a CNN are *hand-crafted* like in classical image processing.  
   **Answer**: False—CNN filters are *learned* during training.

<a name="section2"></a>
## **3. Part I: MNIST — Dense vs. CNN**

<a name="section2_1"></a>
### 3.1 Overview & Loading MNIST


In [ ]:
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.mnist.load_data()
x_train = x_train.astype('float32')/255.0
x_test  = x_test.astype('float32') /255.0

val_size = 5000
x_val = x_train[:val_size]
y_val = y_train[:val_size]
x_train = x_train[val_size:]
y_train = y_train[val_size:]

<a name="section2_2"></a>
### 3.2 Build & Train a Dense Network (Baseline)

In [ ]:
x_train_flat = x_train.reshape(-1, 784)
x_val_flat   = x_val.reshape(-1, 784)
x_test_flat  = x_test.reshape(-1, 784)

model_dense = tf.keras.Sequential([
    tf.keras.layers.Dense(128, activation='relu', input_shape=(784,)),
    tf.keras.layers.Dense(64, activation='relu'),
    tf.keras.layers.Dense(10, activation='softmax')
])

model_dense.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

model_dense.fit(
    x_train_flat, y_train,
    epochs=5, batch_size=64,
    validation_data=(x_val_flat, y_val)
)

test_loss_dense, test_acc_dense = model_dense.evaluate(x_test_flat, y_test)
print("Dense Network Test Accuracy:", test_acc_dense)

<a name="section2_3"></a>
### 3.3 Build & Train a CNN on MNIST

In [ ]:
# Expand dims
x_train_cnn = x_train[..., np.newaxis]
x_val_cnn   = x_val[...,   np.newaxis]
x_test_cnn  = x_test[...,  np.newaxis]

# 2. Define & Compile Model (Functional API)
input_layer = tf.keras.layers.Input(shape=(28,28,1))
x = tf.keras.layers.Conv2D(32, (3,3), activation='relu')(input_layer)
x = tf.keras.layers.MaxPooling2D((2,2))(x)
x = tf.keras.layers.Conv2D(64, (3,3), activation='relu')(x)
x = tf.keras.layers.MaxPooling2D((2,2))(x)
x = tf.keras.layers.Flatten()(x)
x = tf.keras.layers.Dense(64, activation='relu')(x)
output_layer = tf.keras.layers.Dense(10, activation='softmax')(x)

model_cnn = tf.keras.Model(inputs=input_layer, outputs=output_layer)
model_cnn.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

# 3. Train
model_cnn.fit(x_train_cnn, y_train, epochs=5, batch_size=64, validation_data=(x_val_cnn, y_val))

# 4. Evaluate on Test
test_loss, test_acc = model_cnn.evaluate(x_test_cnn, y_test)
print("Test Accuracy:", test_acc)


<a name="section2_4"></a>
### 3.4 Visualizing CNN Filters & Activations

In [ ]:
# 5. Force a forward pass (optional but safe)
_ = model_cnn.predict(x_train_cnn[:1])

# 6. Build Sub-Model
layer_outputs = [layer.output for layer in model_cnn.layers[1:6]]  # Skip Input layer
activation_model = tf.keras.Model(inputs=model_cnn.input, outputs=layer_outputs)

# 7. Check activations on a sample
sample_img = x_test_cnn[:1]
activations = activation_model.predict(sample_img)

# 8. Visualize Only activations[0] and activations[2] (first & second conv layers)
indices_to_show = [0, 2]  # the indexes in 'activations' that correspond to conv layers

for i in indices_to_show:
    layer_activation = activations[i]  # e.g., activations[0] or [2]
    # layer_activation shape -> (1, height, width, channels)

    num_channels = layer_activation.shape[-1]
    n_cols = 8
    n_rows = math.ceil(num_channels / n_cols)

    fig, axes = plt.subplots(n_rows, n_cols, figsize=(n_cols * 1.5, n_rows * 1.5))
    fig.suptitle(f"Layer {i} Activation Maps", fontsize=14)

    # If there's only 1 row, 'axes' might not be a 2D array
    axes = np.array(axes).reshape(n_rows, n_cols)

    for c in range(num_channels):
        row, col = divmod(c, n_cols)
        channel_img = layer_activation[0, :, :, c]
        axes[row, col].imshow(channel_img, cmap='gray')
        axes[row, col].axis('off')

    plt.tight_layout()
    plt.show()


These grids are **activation maps** (feature maps) from the **first** and **second** convolutional layers of your CNN when it sees the digit “7.” Each small image in the grid corresponds to one filter (channel) in that conv layer, showing which parts of the “7” it responds to:

1. **Layer 0 (First Conv Layer)**  
   - You have 32 filters in this layer. Most filters still look a lot like the original digit (though slightly altered), because early filters typically learn to detect basic edges, corners, or strokes.  
   - Bright/white areas in a given sub-image mean that filter sees something it recognizes (like an edge or stroke) at those locations. Dark/black means little or no activation there.

2. **Layer 2 (Second Conv Layer)**  
   - This layer has more filters (64), and each filter tends to be **more specialized**—some respond strongly to diagonal strokes (the slash in “7”), others might pick up smaller corner shapes.  
   - You’ll notice these maps often look more “fragmented” or “abstract.” That’s expected: deeper filters combine or refine the features found by earlier filters, focusing on lines, angles, or partial shapes relevant to the digit.

3. **Why the Differences**  
   - **Early conv layers** (Layer 0) generally highlight simple edges or outlines.  
   - **Deeper conv layers** (Layer 2 here) look for more specific or refined patterns (like diagonals, corners, or intersections).

4. **Interpretation**  
   - A bright region in a feature map means “this filter sees a relevant feature here.”  
   - Some filters appear almost black—meaning they didn’t find that particular “feature” in the digit “7.”  
   - This is how CNNs progressively build up from generic edge detection (Layer 0) to more digit-specific details (Layer 2 and beyond).

**In short:** each small “activation map” shows where that learned filter “fires” for the “7.” The first layer still somewhat resembles the original image with edges highlighted, while the second layer is more abstract, emphasizing diagonal lines or parts of the digit that help the CNN identify it as a “7.”

<a name="section2_5"></a>
### 3.5 **Try-It-Yourself (TIY)**

Below are **code stubs** to guide your modifications:

**TIY #1: Changing Kernel Size**

In [ ]:
# === TIY #1: Insert Your Code Here ===
# Modify (3,3) to (5,5) in the first Conv2D below

# model_cnn_tiy = tf.keras.Sequential([
#     tf.keras.layers.Conv2D(32, (# Insert Your Code Here ), activation='relu', input_shape=(28,28,1)),
#     tf.keras.layers.MaxPooling2D((2,2)),
#     tf.keras.layers.Conv2D(64, (3,3), activation='relu'),
#     tf.keras.layers.MaxPooling2D((2,2)),
#     tf.keras.layers.Flatten(),
#     tf.keras.layers.Dense(64, activation='relu'),
#     tf.keras.layers.Dense(10, activation='softmax')
# ])

# model_cnn_tiy.compile(...)
# model_cnn_tiy.fit(...)

**TIY #2: Add/Remove a Conv Layer**

In [ ]:
# === TIY #2: Insert Your Code Here ===
# You can add an extra Conv2D + MaxPooling2D block or remove one.
# Watch for overfitting or changes in accuracy & training time.

**TIY #3: Visualize Misclassifications**

In [ ]:
# === TIY #3: Insert Your Code Here ===
# 1. preds = model_cnn.predict(x_test_cnn)
# 2. predicted_labels = np.argmax(preds, axis=1)
# 3. Compare with y_test, plot some misclassified digits.

<a name="section2_6"></a>
### 3.6 Mini-Quiz #2

1. *Why might a CNN outperform a dense network on MNIST?*  
   - **Answer**: Because convolution captures local pixel patterns with fewer parameters.

2. *Which layer typically follows a Conv2D layer to reduce spatial dimensions?*  
   - A) **MaxPooling2D**  
   - B) Dense  
   - C) Activation

<a name="section3"></a>
## **4. Part II: CIFAR-10 — Data Augmentation**

<a name="section3_1"></a>
### 4.1 Loading & Exploring CIFAR-10

In [ ]:
(x_train_c10, y_train_c10), (x_test_c10, y_test_c10) = tf.keras.datasets.cifar10.load_data()
x_train_c10 = x_train_c10.astype('float32') / 255.0
x_test_c10  = x_test_c10.astype('float32')  / 255.0

val_size = 5000
x_val_c10 = x_train_c10[:val_size]
y_val_c10 = y_train_c10[:val_size]
x_train_c10 = x_train_c10[val_size:]
y_train_c10 = y_train_c10[val_size:]

**Classes**: airplane, automobile, bird, cat, deer, dog, frog, horse, ship, truck.

<a name="section3_1_1"></a>
### 4.1.1 **Data Augmentation Demonstration**

In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import matplotlib.pyplot as plt

demo_datagen = ImageDataGenerator(
    rotation_range=15,
    width_shift_range=0.1,
    height_shift_range=0.1,
    horizontal_flip=True
)

sample_image_c10 = x_train_c10[0]
sample_label_c10 = y_train_c10[0]

demo_batch = np.expand_dims(sample_image_c10, axis=0)
demo_iter = demo_datagen.flow(demo_batch, batch_size=1)

fig, axs = plt.subplots(1, 5, figsize=(15,3))
for i in range(5):
    aug_img = next(demo_iter)[0].astype('float32')
    axs[i].imshow(aug_img)
    axs[i].axis('off')
axs[0].set_title(f"Original label: {sample_label_c10[0]}")
plt.show()

We’re seeing **randomly augmented versions** of the **same CIFAR-10 image** via our `ImageDataGenerator` settings. Here’s what’s happening in each step:

1. **Single Base Image**  
   - You took `sample_image_c10 = x_train_c10[0]` (with label 6, meaning “frog” in CIFAR‐10).  
   - By expanding dims and creating `demo_iter = demo_datagen.flow(...)`, you feed the same single image into the augmentation pipeline over and over.

2. **Random Transformations**  
   - The `ImageDataGenerator` parameters  
     ```python
     rotation_range=15,
     width_shift_range=0.1,
     height_shift_range=0.1,
     horizontal_flip=True
     ```
     apply random:
     - **Rotations** up to 15 degrees,  
     - **Horizontal flips** with 50% probability,  
     - **Translations** (shifts) horizontally or vertically by up to 10% of the image width/height.  
   - Each time you call `next(demo_iter)`, you get a **new** random transformation of that same frog image.

3. **Result**  
   - The 5 subplots show slightly **rotated**, **shifted**, or possibly **flipped** versions of the frog.  
   - Even though the image is the “same frog,” each augmented copy looks different: some may be shifted left/right, some rotated a bit, and any might be horizontally flipped.

4. **Why This Matters**  
   - In training, each epoch sees many augmented variations of the same original image, making the model more **robust** to minor translations, flips, and rotations.  
   - This reduces overfitting, teaching the CNN that small orientation or position changes shouldn’t alter the class label.



<a name="section3_2"></a>
### 4.2 CNN with Data Augmentation

In [ ]:
train_datagen = ImageDataGenerator(
    rotation_range=15,
    width_shift_range=0.1,
    height_shift_range=0.1,
    horizontal_flip=True
)
val_datagen = ImageDataGenerator()

batch_size = 64
train_generator_c10 = train_datagen.flow(x_train_c10, y_train_c10, batch_size=batch_size)
val_generator_c10   = val_datagen.flow(x_val_c10, y_val_c10, batch_size=batch_size)

model_cifar = tf.keras.Sequential([
    tf.keras.layers.Conv2D(32, (3,3), activation='relu', input_shape=(32,32,3)),
    tf.keras.layers.MaxPooling2D((2,2)),
    tf.keras.layers.Dropout(0.25),

    tf.keras.layers.Conv2D(64, (3,3), activation='relu'),
    tf.keras.layers.MaxPooling2D((2,2)),
    tf.keras.layers.Dropout(0.25),

    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(128, activation='relu'),
    tf.keras.layers.Dropout(0.5),
    tf.keras.layers.Dense(10, activation='softmax')
])

model_cifar.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

history_cifar = model_cifar.fit(
    train_generator_c10,
    steps_per_epoch=len(x_train_c10)//batch_size,
    epochs=10,
    validation_data=val_generator_c10,
    validation_steps=len(x_val_c10)//batch_size
)

test_loss_cifar, test_acc_cifar = model_cifar.evaluate(x_test_c10, y_test_c10, batch_size=batch_size)
print("CIFAR-10 Test Accuracy:", test_acc_cifar)

<a name="section3_3"></a>
### 4.3 **Try-It-Yourself (TIY)**

**TIY #4: Adjust Augmentations**

In [ ]:
# === TIY #4: Insert Your Code Here ===
# 1. Add brightness_range, zoom_range, or shear_range to train_datagen
# 2. Re-train, observe changes in final accuracy

**TIY #5: Confusion Matrix**

In [ ]:
# === TIY #5: Insert Your Code Here ===
# 1. preds = model_cifar.predict(x_test_c10)
# 2. predicted_classes = np.argmax(preds, axis=1)
# 3. true_classes = y_test_c10.flatten()
# 4. Use sklearn.metrics.confusion_matrix, plot a heatmap

<a name="section3_4"></a>
### 4.4 Mini-Quiz #3

1. *Which function in Keras helps us apply random flips, shifts, or rotations to images?*  
   - A) ImageDataGenerator  
   - B) data.DataLoader  
   - C) flow_from_directory  

2. *True or False?* Data augmentation can reduce overfitting by providing diverse training samples.  
   - **Answer**: True

<a name="section4"></a>
## **5. Part III: Transfer Learning (Cats vs. Dogs)**

<a name="section4_1"></a>
### 5.1 Why Transfer Learning?
- Use a **pretrained** CNN on ImageNet.  
- **Feature Extraction**: Freeze base layers.  
- **Fine-Tuning**: Unfreeze top layers to adapt them to your dataset.

Here we’ll use **VGG16** pretrained on ImageNet to classify Cats vs. Dogs. First, we’ll do **feature extraction** with the convolutional base **frozen**. Then we’ll do **fine-tuning** of the top layers. Lastly, we’ll visualize the **architecture** to see how VGG16 is structured internally.

<a name="section4_2"></a>
### Loading & Preprocessing Cats vs. Dogs

In [ ]:
import tensorflow as tf
import tensorflow_datasets as tfds

(ds_full, ds_info) = tfds.load('cats_vs_dogs', with_info=True, as_supervised=True)
ds_train = tfds.load('cats_vs_dogs', split='train[:80%]', as_supervised=True)
ds_val   = tfds.load('cats_vs_dogs', split='train[80%:90%]', as_supervised=True)
ds_test  = tfds.load('cats_vs_dogs', split='train[90%:]',   as_supervised=True)

def format_image(image, label):
    image = tf.image.resize(image, (224,224))
    image = image / 255.0
    return image, label

batch_size = 32
ds_train = ds_train.map(format_image).shuffle(1000).batch(batch_size).prefetch(tf.data.AUTOTUNE)
ds_val   = ds_val.map(format_image).batch(batch_size).prefetch(tf.data.AUTOTUNE)
ds_test  = ds_test.map(format_image).batch(batch_size).prefetch(tf.data.AUTOTUNE)

### **5.2 Visualizing the VGG16 Base**

Before we do any training, let’s load **VGG16** (convolutional base) and inspect its architecture. Two methods:  
1. **Model Summary** (`.summary()`)  
2. **Plot Model** (`tf.keras.utils.plot_model()`)

In [ ]:
from tensorflow.keras.applications import VGG16
from tensorflow.keras.utils import plot_model

base_model = VGG16(weights='imagenet', include_top=False, input_shape=(224,224,3))

# Print architecture text
base_model.summary()

# Optionally create a visual diagram (requires pydot/graphviz installed)
# plot_model(base_model, to_file='vgg16_base.png', show_shapes=True)

**Explanation**:
- `include_top=False` loads only the convolutional layers (no fully connected head).  
- `input_shape=(224,224,3)` means it expects 224×224 RGB images.  
- The `.summary()` shows each layer’s name, output shape, and number of parameters.  
- The `plot_model(...)` call can produce a PNG diagram with layer connections if your environment has Graphviz installed.

<a name="section4_3"></a>
### 5.3 Feature Extraction with VGG16

In feature extraction mode, we **freeze** the convolutional base so we don’t alter its ImageNet weights, then add new layers for binary classification (cats vs. dogs).

In [ ]:
from tensorflow.keras import layers, models

# Freeze all layers in base_model
for layer in base_model.layers:
    layer.trainable = False

# Build the final model
model_tl = models.Sequential([
    base_model,
    layers.Flatten(),
    layers.Dense(256, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(1, activation='sigmoid')  # binary classification
])

model_tl.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

model_tl.summary()  # see entire model, including new layers

# Train ~5 epochs
model_tl.fit(ds_train, epochs=5, validation_data=ds_val)

# Evaluate
test_loss, test_acc = model_tl.evaluate(ds_test)
print("Feature Extraction - Test Accuracy:", test_acc)

**Key Points**:
- `layer.trainable = False` ensures we do **not** update the VGG16 conv weights.  
- We add our own Flatten → Dense(256) → Dropout → Dense(1) for the final classification.  
- Typically, you can get decent accuracy quickly since the conv base already knows a lot about edges/textures from ImageNet.


### **5.4 Fine-Tuning the Top Layers**

After feature extraction, we can **unfreeze** some top convolutional layers to adapt them to our Cats vs. Dogs dataset. Usually, we do this with a **smaller learning rate**.

In [ ]:
# Unfreeze the last 4 layers
for layer in base_model.layers[-4:]:
    layer.trainable = True

model_tl.compile(
    optimizer=tf.keras.optimizers.Adam(1e-5),  # smaller LR
    loss='binary_crossentropy',
    metrics=['accuracy']
)

model_tl.fit(ds_train, epochs=5, validation_data=ds_val)

test_loss, test_acc = model_tl.evaluate(ds_test)
print("Fine-Tuning - Test Accuracy:", test_acc)

**Why smaller LR?**  
- The rest of the layers are from a pretrained conv base. We don’t want to drastically alter them, just fine‐tune the top layers to better fit cats/dogs if the data is sufficient.

<a name="section4_4"></a>
### 5.5 **Try-It-Yourself (TIY)**


**TIY #6: Freeze vs. Unfreeze**

In [ ]:
# === TIY #6: Insert Your Code Here ===
# 1. Freeze entire base -> train 3-5 epochs
# 2. Unfreeze last 4 layers -> smaller LR -> re-train
# 3. Compare final accuracy

**TIY #7: Change Pretrained Model**

In [ ]:
# === TIY #7: Insert Your Code Here ===
# from tensorflow.keras.applications import ResNet50 or MobileNetV2
# base_model_2 = ResNet50(...)
# Compare performance vs. VGG16

**TIY #8: Visualize Misclassifications**

In [ ]:
# === TIY #8: Insert Your Code Here ===
# 1. preds = model_tl.predict(ds_test)
# 2. threshold -> 0 or 1
# 3. gather true_labels, compare, plot misclassified cat/dog images

<a name="section4_5"></a>
### 5.6 Mini-Quiz #4

1. *When we unfreeze top layers in transfer learning, we should typically?*  
   - A) **Use a smaller learning rate**  
   - B) Use a bigger learning rate  
   - C) Use the same learning rate  

2. *True or False?* Feature extraction alone can’t achieve high accuracy.  
   - **Answer**: False—often it’s good enough, but fine-tuning can improve results further.

<a name="section5"></a>
## **6. Optional: TensorBoard Demonstration**

**Why**: Visualizing training/validation curves can help debug or monitor training progress.

In [ ]:
from tensorflow.keras.callbacks import TensorBoard
import datetime

log_dir = "logs/fit/" + datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
tensorboard_callback = TensorBoard(log_dir=log_dir, histogram_freq=1)

# Example usage in model_cifar or model_cnn:
model_cnn.fit(
    x_train_cnn, y_train,
    epochs=5, batch_size=64,
    validation_data=(x_val_cnn, y_val),
    callbacks=[tensorboard_callback]
)

# In Colab:
%load_ext tensorboard
%tensorboard --logdir logs/fit

<a name="section6"></a>
## **7. Optional: Grad-CAM Demonstration**

**Goal**: See *where* the CNN “looked” in an image.

In [ ]:
import tensorflow as tf
import tensorflow_datasets as tfds
import numpy as np
import matplotlib.pyplot as plt
import cv2

##############################################################
# 1) Data Loading
##############################################################
(ds_full, ds_info) = tfds.load(
    'cats_vs_dogs',
    with_info=True,
    as_supervised=True
)
ds_train = tfds.load('cats_vs_dogs', split='train[:80%]', as_supervised=True)
ds_val   = tfds.load('cats_vs_dogs', split='train[80%:90%]', as_supervised=True)
ds_test  = tfds.load('cats_vs_dogs', split='train[90%:]',   as_supervised=True)

def format_image(image, label):
    image = tf.image.resize(image, (224,224))
    image = image / 255.0
    return image, label

batch_size = 16
ds_train = (
    ds_train
    .map(format_image)
    .shuffle(1000)
    .batch(batch_size)
    .prefetch(tf.data.AUTOTUNE)
)
ds_val = (
    ds_val
    .map(format_image)
    .batch(batch_size)
    .prefetch(tf.data.AUTOTUNE)
)
ds_test = (
    ds_test
    .map(format_image)
    .batch(batch_size)
    .prefetch(tf.data.AUTOTUNE)
)

##############################################################
# 2) Build VGG16 Feature Extractor
##############################################################
from tensorflow.keras.applications import VGG16
from tensorflow.keras import layers, models

# 2a) Load VGG16 (include_top=False => no Dense layers, up to block5_conv3)
base_model = VGG16(weights='imagenet',
                   include_top=False,
                   input_shape=(224,224,3))

# Freeze its layers
for layer in base_model.layers:
    layer.trainable = False

# Our feature extractor: input -> base_model -> outputs from last conv layer
# (In VGG16, the last conv is indeed "block5_conv3".)
# But since we used include_top=False, base_model already ends at "block5_pool".
# We'll just take the base_model as-is if we want the final pooling (7x7).
# If you specifically want block5_conv3, you could slice the model at that layer:
# last_conv_layer = base_model.get_layer("block5_conv3")
# feature_extractor = tf.keras.Model(inputs=base_model.input,
#                                    outputs=last_conv_layer.output)
#
# But let's keep the default final layer which is "block5_pool" -> shape (7,7,512)
# to be consistent. The difference is just one pooling operation.

feature_extractor = base_model  # We'll use the entire base_model for conv features.

##############################################################
# 3) Build Classifier Head
##############################################################
# Check the shape of feature maps from base_model
dummy_input = tf.zeros((1, 224, 224, 3))
dummy_output = feature_extractor(dummy_input)
print("Shape of base_model output:", dummy_output.shape)
# Typically (1, 7, 7, 512) for 224x224 input

feature_shape = dummy_output.shape[1:]  # (7, 7, 512)

# Build a simple 2-layer classifier
classifier_input = layers.Input(shape=feature_shape)
x = layers.Flatten()(classifier_input)
x = layers.Dense(256, activation='relu')(x)
x = layers.Dropout(0.5)(x)
classifier_output = layers.Dense(1, activation='sigmoid')(x)

classifier_head = models.Model(
    inputs=classifier_input,
    outputs=classifier_output,
    name="classifier_head"
)

##############################################################
# 4) Combine for Training
##############################################################
# We'll make a single top-level model:
#   input -> feature_extractor -> classifier_head
full_input = layers.Input(shape=(224, 224, 3))
features = feature_extractor(full_input)        # shape (None,7,7,512)
predictions = classifier_head(features)         # shape (None,1)

model_tl = models.Model(full_input, predictions, name="VGG16_TL")

model_tl.compile(optimizer='adam',
                 loss='binary_crossentropy',
                 metrics=['accuracy'])

# Train & Evaluate
model_tl.fit(ds_train, epochs=3, validation_data=ds_val)
test_loss, test_acc = model_tl.evaluate(ds_test)
print("Initial Test Accuracy:", test_acc)

In [ ]:
##############################################################
# 5) Grad-CAM with the Split-Model Approach
##############################################################
def make_gradcam_heatmap(img_array, feature_extractor, classifier_head):
    """
    Generate Grad-CAM heatmap for a single image (or batch).

    img_array          : (1,224,224,3)
    feature_extractor  : model returning convolutional features
    classifier_head    : model returning final predictions from features
    """
    with tf.GradientTape() as tape:
        # 1) Forward pass: get conv features
        conv_outputs = feature_extractor(img_array)
        # 2) Watch these outputs
        tape.watch(conv_outputs)

        # 3) Forward pass through classifier
        predictions = classifier_head(conv_outputs)

        # For binary classification, we consider the single logit output
        class_channel = predictions[:, 0]

    # 4) Compute gradients
    grads = tape.gradient(class_channel, conv_outputs)

    # 5) Global-average-pool the gradients over the spatial dimensions
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))

    # 6) Build the heatmap as a channel-wise weighted sum of conv_outputs
    conv_outputs = conv_outputs[0]  # shape e.g. (7,7,512)
    heatmap = tf.reduce_sum(conv_outputs * pooled_grads, axis=-1)

    # 7) ReLU and normalize
    heatmap = np.maximum(heatmap, 0)
    heatmap /= (heatmap.max() + 1e-10)

    return heatmap

def overlay_heatmap(heatmap, original_img, alpha=0.4):
    """
    Overlays heatmap on the original_img (both in uint8).
    """
    import matplotlib.cm as cm

    # 1) Convert the 2D heatmap (range [0..1]) to [0..255] uint8
    heatmap_uint8 = np.uint8(255 * heatmap)

    # 2) Apply the JET colormap. This returns an RGBA array in float [0..1].
    jet_colors = cm.jet(heatmap_uint8)[:, :, :3]  # drop the alpha channel

    # 3) Scale the color map up to [0..255], cast to uint8
    jet_colors = np.uint8(255 * jet_colors)

    # 4) Resize to match original image size
    jet_colors = cv2.resize(
        jet_colors,
        (original_img.shape[1], original_img.shape[0])
    )

    # 5) Convert from RGB to BGR if needed
    heatmap_bgr = cv2.cvtColor(jet_colors, cv2.COLOR_RGB2BGR)

    # 6) AddWeighted requires both inputs to have the same type (uint8).
    # original_img is uint8 (assuming you scaled it by 255 before).
    superimposed_img = cv2.addWeighted(
        original_img, 1 - alpha,
        heatmap_bgr, alpha,
        0
    )
    return superimposed_img


##############################################################
# 6) Grad-CAM Example
##############################################################
# Grab 1 image/label from ds_test
for images, labels in ds_test.take(1):
    cat_dog_img = images[:1]   # shape (1,224,224,3)
    cat_dog_label = labels[0]  # single label
    break

# 1) Generate heatmap
heatmap = make_gradcam_heatmap(
    cat_dog_img,
    feature_extractor,
    classifier_head
)

# 2) Convert single image to uint8 for overlay
orig_img_uint8 = np.uint8(cat_dog_img[0].numpy() * 255)

# 3) Overlay
superimposed_img = overlay_heatmap(heatmap, orig_img_uint8, alpha=0.4)

# 4) Plot the original and the Grad-CAM
plt.figure(figsize=(10,4))

plt.subplot(1,2,1)
plt.imshow(orig_img_uint8)
plt.title(f"Original (label={cat_dog_label.numpy()})")
plt.axis('off')

plt.subplot(1,2,2)
# Convert BGR -> RGB for display
overlay_rgb = cv2.cvtColor(superimposed_img, cv2.COLOR_BGR2RGB)
plt.imshow(overlay_rgb)
plt.title("Grad-CAM")
plt.axis('off')

plt.show()


<a name="section7"></a>
## **8. Hyperparameter Tuning Tips**

1. **Manual**: Try different batch sizes, learning rates, kernel sizes. Document results.  
2. **Keras Tuner**:  
   ```python
   # pip install keras-tuner
   # Example usage of a tuner with different hyperparameters.
   ```
3. **Observe** if bigger networks or more layers actually help or cause overfitting.

<a name="section8"></a>
## **9. Reflection & Summary**

**Students Learned**:
1. How **convolutions** detect edges and shapes (manual filter demo).  
2. **Dense vs. CNN** on MNIST, seeing CNN’s advantage.  
3. **Data Augmentation** on CIFAR-10 (demonstration + training).  
4. **Transfer Learning** for Cats vs. Dogs, with feature extraction & optional fine-tuning.  
5. **TensorBoard** for monitoring, **Grad-CAM** for interpretability, and **basic hyperparameter tuning** tips.

<a name="section9"></a>
## **10. Additional Best Practices**

- **Experiment Logging**: Keep a record (spreadsheet/MLflow) of changes & results.  
- **Early Stopping**: If validation loss stops improving, halt training.  
- **Data Splits**: Always maintain a separate test set to avoid data leakage.


<a name="section10"></a>
## **11. Common Pitfalls & Solutions**

1. **Overfitting**: More augmentation, dropout, or smaller architecture.  
2. **Shape Errors**: Always confirm `(batch, height, width, channels)`.  
3. **Learning Rate**: If stuck, lower or raise LR, or try a scheduler.  
4. **Data Leakage**: Strictly separate train/val/test.

<a name="section11"></a>
## **12. Wrap-Up & Further Resources**

**Congratulations**:  
- You’ve completed a **full journey**: from *manual convolution* to *CNN training* on MNIST/CIFAR-10, then *transfer learning* for Cats vs. Dogs.  
- **Try** adding more advanced augmentation (RandAugment, AutoAugment), or **Grad-CAM** for model interpretability.  
- Tinker with **hyperparameter tuning** for even better results.

### **Further Resources**

1. [Keras Documentation](https://keras.io/)  
2. [TensorFlow Tutorials](https://www.tensorflow.org/tutorials)  
3. [Stanford CS231n](http://cs231n.stanford.edu/)  
4. [Andrew Ng’s Deep Learning Specialization](https://www.coursera.org/specializations/deep-learning)